In [1]:
# E-commerce Sales & Traffic Analysis — Final Project

### Важливі посилання на матеріали проєкту:
### Інтерактивний Tableau Dashboard (2 сторінки): [Переглянути на Tableau Public](https://public.tableau.com/app/profile/mykyta.yemelianov3525/viz/Book1_17583094938840/UserTrafficDashboard?publish=yes)
### Повний GitHub-репозиторій проєкту:** [Переглянути на GitHub](https://github.com/mykytaggwp/portfolio_da)

# 🛒 E-commerce Sales & User Traffic Analytics

## Крок 1. Підключення та вивантаження даних з Google BigQuery
Вивантаження об'єднаного датасету (сесії, параметри пристроїв, акаунти, замовлення та товари) за допомогою SQL-запиту та Google BigQuery API.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from google.cloud import bigquery

# Вимикаємо інформаційні попередження Google Cloud
warnings.filterwarnings('ignore', category=UserWarning, module='google.auth')

# SQL-запит для вивантаження даних
sql_query = """
SELECT 
    s.date AS session_date,
    s.ga_session_id,
    sp.continent,
    sp.country,
    sp.device AS device_type,
    sp.browser,
    sp.mobile_model_name AS device_model,
    sp.operating_system,
    sp.language AS browser_language,
    sp.medium AS traffic_source,
    sp.channel AS traffic_channel,
    a.id AS user_id,
    a.is_verified AS is_email_confirmed,
    a.is_unsubscribed,
    p.category AS product_category,
    p.name AS product_name,
    p.price,
    p.short_description AS product_description
FROM `data-analytics-mate.DA.session` AS s
LEFT JOIN `data-analytics-mate.DA.session_params` AS sp ON s.ga_session_id = sp.ga_session_id
LEFT JOIN `data-analytics-mate.DA.account_session` AS ans ON s.ga_session_id = ans.ga_session_id
LEFT JOIN `data-analytics-mate.DA.account` AS a ON ans.account_id = a.id
LEFT JOIN `data-analytics-mate.DA.order` AS o ON s.ga_session_id = o.ga_session_id
LEFT JOIN `data-analytics-mate.DA.product` AS p ON o.item_id = p.item_id;
"""

# Виконання запиту напряму через BigQuery API
try:
    client = bigquery.Client(project='data-analytics-mate')
    df = client.query(sql_query).to_dataframe()
    print(f"Дані успішно завантажено безпосередньо з Google BigQuery! Рядків: {len(df)}")
except Exception as e:
    print(f"Помилка прямого запиту: {e}")
    print("Завантаження даних із локального файлу portfolio_dataset.csv...")
    df = pd.read_csv('portfolio_dataset.csv')

df.head()

C:\Users\mykyt\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## Крок 2. Опис отриманого датасету (Data Overview)
Аналіз структури даних, типів колонок, розмірності датасету, підрахунок унікальних сесій, часового діапазону та виявлення пропущених значень.

In [ ]:
df['session_date'] = pd.to_datetime(df['session_date'])

total_rows, total_cols = df.shape

datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

unique_sessions = df['ga_session_id'].nunique()
min_date = df['session_date'].min().strftime('%Y-%m-%d')
max_date = df['session_date'].max().strftime('%Y-%m-%d')

null_counts = df.isnull().sum()
null_pct = (null_counts / total_rows) * 100

null_summary = pd.DataFrame({
    'Колонка': null_counts.index,
    'Кількість пропусків': null_counts.values,
    'Відсоток пропусків (%)': null_pct.round(2).values
})
null_summary = null_summary[null_summary['Кількість пропусків'] > 0].reset_index(drop=True)

print(f"--- ЗАГАЛЬНИЙ ОГЛЯД ---")
print(f"Загальна кількість рядків: {total_rows}")
print(f"Загальна кількість колонок: {total_cols}\n")

print(f"--- РОЗПОДІЛ ЗА ТИПАМИ ДАНИХ ---")
print(f"Колонки типу datetime ({len(datetime_cols)}): {datetime_cols}")
print(f"Колонки числового типу ({len(numeric_cols)}): {numeric_cols}")
print(f"Колонки категоріального типу ({len(categorical_cols)}): {categorical_cols}\n")

print(f"--- МЕТРИКИ СЕСІЙ ТА ЧАСУ ---")
print(f"Кількість унікальних сесій: {unique_sessions}")
print(f"Часовий період: від {min_date} до {max_date}\n")

print("--- ПРОПУЩЕНІ ЗНАЧЕННЯ ---")
print(null_summary.to_string(index=False))

> **Висновки по датасету:**
> * Датасет містить **349,545** рядків та охоплює період з **01.11.2020 по 31.01.2021** (3 місяці).
> * Пропуски у стовпцях акаунтів (`user_id`, `is_email_confirmed`) та замовлень (`price`, `product_category`) є природними, оскільки не кожна сесія супроводжується авторизацією або покупкою.

## Крок 3. Розвідувальний аналіз даних (Exploratory Data Analysis — EDA)
Дослідження ключових факторів виручки: географія ринків, топ-категорії товарів, розподіл трафіку за каналами та типами пристроїв.

In [ ]:
sales_by_continent = df.groupby('continent').agg(
    total_sales=('price', 'sum'),
    orders_count=('price', 'count')
).sort_values(by='total_sales', ascending=False).head(3)

sales_by_country = df.groupby('country').agg(
    total_sales=('price', 'sum'),
    orders_count=('price', 'count')
).sort_values(by='total_sales', ascending=False).head(5)

sales_by_continent['total_sales'] = sales_by_continent['total_sales'].map('{:,.2f} $'.format)
sales_by_country['total_sales'] = sales_by_country['total_sales'].map('{:,.2f} $'.format)

print("--- ТОП-3 КОНТИНЕНТИ ЗА ПРОДАЖАМИ ТА КІЛЬКІСТЮ ЗАМОВЛЕНЬ ---")
print(sales_by_continent.to_string())

print("\n--- ТОП-5 КРАЇН ЗА ПРОДАЖАМИ ТА КІЛЬКІСТЮ ЗАМОВЛЕНЬ ---")
print(sales_by_country.to_string())

In [ ]:
top10_global = df.groupby('product_category')['price'].sum().nlargest(10)

top_country = df.groupby('country')['price'].sum().idxmax()

top10_top_country = df[df['country'] == top_country].groupby('product_category')['price'].sum().nlargest(10)

category_comparison = pd.DataFrame({
    'Глобальні продажі ($)': top10_global.map('{:,.2f}'.format),
    f'Продажі у {top_country} ($)': top10_top_country.map('{:,.2f}'.format)
})

print("--- ТОП-10 КАТЕГОРІЙ ТОВАРІВ: ГЛОБАЛЬНО VS КРАЇНА-ЛІДЕР ---")
print(category_comparison.to_string())

In [ ]:
total_revenue = df['price'].sum()

sales_by_device = (df.groupby('device_type')['price'].sum() / total_revenue * 100).round(2).reset_index()
sales_by_device.columns = ['Тип девайса', 'Частка продажів (%)']
sales_by_device = sales_by_device.sort_values(by='Частка продажів (%)', ascending=False)

sales_by_channel = (df.groupby('traffic_channel')['price'].sum() / total_revenue * 100).round(2).reset_index()
sales_by_channel.columns = ['Канал трафіку', 'Частка продажів (%)']
sales_by_channel = sales_by_channel.sort_values(by='Частка продажів (%)', ascending=False)

sales_by_source = (df.groupby('traffic_source')['price'].sum() / total_revenue * 100).round(2).reset_index()
sales_by_source.columns = ['Джерело трафіку', 'Частка продажів (%)']
sales_by_source = sales_by_source.sort_values(by='Частка продажів (%)', ascending=False).head(5)

print("--- ПРОДАЖІ ЗА ТИПАМИ ДЕВАЙСІВ (%) ---")
print(sales_by_device.to_string(index=False))

print("\n--- ПРОДАЖІ ЗА КАНАЛАМИ ТРАФІКУ (%) ---")
print(sales_by_channel.to_string(index=False))

print("\n--- ТОП-5 ДЖЕРЕЛ ТРАФІКУ ЗА ПРОДАЖАМИ (%) ---")
print(sales_by_source.to_string(index=False))

In [ ]:
reg_df = df[df['user_id'].notnull()]

unique_users = reg_df.drop_duplicates(subset=['user_id'])
total_unique_users = len(unique_users)

pct_email_confirmed = (unique_users['is_email_confirmed'].mean() * 100).round(2)

pct_unsubscribed = (unique_users['is_unsubscribed'].mean() * 100).round(2)

behavior_comp = reg_df.groupby('is_unsubscribed').agg(
    total_sales=('price', 'sum'),
    orders_count=('price', 'count'),
    avg_order_value=('price', 'mean'),
    unique_users=('user_id', 'nunique')
).reset_index()

behavior_comp['is_unsubscribed'] = behavior_comp['is_unsubscribed'].map({0.0: 'Підписані', 1.0: 'Відписалися'})
behavior_comp['sales_per_user'] = (behavior_comp['total_sales'] / behavior_comp['unique_users']).round(2)

top5_reg_countries = unique_users['country'].value_counts().head(5).reset_index()
top5_reg_countries.columns = ['Країна', 'Кількість користувачів']

print(f"--- ЗАГАЛЬНА СТАТИСТИКА КОРИСТУВАЧІВ ---")
print(f"Усього унікальних зареєстрованих користувачів: {total_unique_users}")
print(f"Частка користувачів з підтвердженим email: {pct_email_confirmed}%")
print(f"Частка користувачів, які відписалися від розсилки: {pct_unsubscribed}%\n")

print("--- ПОРІВНЯННЯ ПОВЕДІНКИ ПІДПИСАНИХ ТА ВІДПИСАНИХ КОРИСТУВАЧІВ ---")
print(behavior_comp.to_string(index=False))

print("\n--- ТОП-5 КРАЇН ЗА КІЛЬКІСТЮ ЗАРЕЄСТРОВАНИХ КОРИСТУВАЧІВ ---")
print(top5_reg_countries.to_string(index=False))

> **Ключові EDA-інсайти:**
> * **Категорії:** Найбільшу виручку генерує категорія **Sofas & armchairs**, за нею — **Chairs** та **Beds**.
> * **Географія:** Домінуючим ринком за обсягом продажів є **United States**.
> * **Канали та Девайси:** Основний обсяг сесій генерують **Organic** та **Paid Search**, а за виручкою абсолютним лідером є **Desktop**.

## Крок 4. Динаміка продажів та сезонність (Sales Trends & Time Series)
Аналіз щоденних та агрегованих часових рядів продажів для виявлення сезонних коливань і піків попиту.

In [ ]:
# 1. Перетворення дати та розрахунок щоденних продажів
df['session_date'] = pd.to_datetime(df['session_date'])
daily_sales = df.groupby('session_date')['price'].sum().reset_index()

# 2. Побудова графіка
plt.figure(figsize=(14, 5))
ax = sns.lineplot(data=daily_sales, x='session_date', y='price', marker='o', color='#1f77b4', linewidth=2)

plt.title('Загальна щоденна динаміка продажів (01.11.2020 – 31.01.2021)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Загальні продажі ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

# Форматування осі X: підпис раз на 7 днів
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%Y'))
plt.xticks(rotation=45)

# Форматування осі Y у тисячах для зручності читання
ax.yaxis.set_major_formatter('{x:,.0f} $')

plt.tight_layout()
plt.show()

# 3. Форматування та вивід таблиці (head/tail замість усіх 92 рядків)
daily_sales_display = daily_sales.copy()
daily_sales_display['День тижня'] = daily_sales['session_date'].dt.day_name()
daily_sales_display['session_date'] = daily_sales_display['session_date'].dt.strftime('%Y-%m-%d')
daily_sales_display['price'] = daily_sales_display['price'].map('{:,.2f} $'.format)
daily_sales_display = daily_sales_display[['session_date', 'price', 'День тижня']]
daily_sales_display.columns = ['Дата', 'Продажі ($)', 'День тижня']

print("--- ЩОДЕННІ ПРОДАЖІ ТА ДНІ ТИЖНЯ (Перші 5 та останні 5 днів) ---")
print(pd.concat([daily_sales_display.head(5), daily_sales_display.tail(5)]).to_string(index=False))

In [ ]:
top_continents = ['Americas', 'Asia', 'Europe']
df_continents = df[df['continent'].isin(top_continents)].copy()

# 1. Агрегація за датою та континентом
continent_daily = df_continents.groupby(['session_date', 'continent'])['price'].sum().reset_index()

# 2. Побудова графіка
plt.figure(figsize=(14, 6))
palette = {'Americas': '#2b5c8f', 'Asia': '#e07a5f', 'Europe': '#81b29a'}

ax = sns.lineplot(
    data=continent_daily, 
    x='session_date', 
    y='price', 
    hue='continent', 
    palette=palette, 
    linewidth=2.2, 
    marker='o',
    markersize=4
)

plt.title('Динаміка продажів за континентами (Americas, Asia, Europe)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Загальні продажі ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Континент', title_fontsize='11', fontsize='10')

# Форматування осі X: підписи кожні 7 днів
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%Y'))
plt.xticks(rotation=45)

# Форматування осі Y
ax.yaxis.set_major_formatter('{x:,.0f} $')

plt.tight_layout()
plt.show()

# 3. Зведена таблиця
pivot_continents = continent_daily.pivot(index='session_date', columns='continent', values='price').fillna(0)
pivot_continents.index = pivot_continents.index.strftime('%Y-%m-%d')
pivot_continents_formatted = pivot_continents.apply(lambda col: col.map('{:,.2f} $'.format))

print("--- ПРОДАЖІ ЗА КОНТИНЕНТАМИ ПО ДНЯХ (Перші 5 та останні 5 днів) ---")
print(pd.concat([pivot_continents_formatted.head(5), pivot_continents_formatted.tail(5)]))

In [ ]:
# 1. Агрегація за датою та каналом трафіку
channel_daily = df.groupby(['session_date', 'traffic_channel'])['price'].sum().reset_index()

# 2. Побудова графіка
plt.figure(figsize=(14, 6))

ax = sns.lineplot(
    data=channel_daily, 
    x='session_date', 
    y='price', 
    hue='traffic_channel', 
    linewidth=2.2, 
    marker='o',
    markersize=4
)

plt.title('Динаміка продажів за каналами трафіку (Organic, Paid, Direct та ін.)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Загальні продажі ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Канал трафіку', title_fontsize='11', fontsize='10')

# Форматування осі X: підписи раз на 7 днів
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%Y'))
plt.xticks(rotation=45)

# Форматування осі Y
ax.yaxis.set_major_formatter('{x:,.0f} $')

plt.tight_layout()
plt.show()

# 3. Зведена таблиця (перші 5 та останні 5 днів)
pivot_channels = channel_daily.pivot(index='session_date', columns='traffic_channel', values='price').fillna(0)
pivot_channels.index = pivot_channels.index.strftime('%Y-%m-%d')
pivot_channels_formatted = pivot_channels.apply(lambda col: col.map('{:,.2f} $'.format))

print("--- ПРОДАЖІ ЗА КАНАЛАМИ ТРАФІКУ ПО ДНЯХ (Перші 5 та останні 5 днів) ---")
print(pd.concat([pivot_channels_formatted.head(5), pivot_channels_formatted.tail(5)]))

In [ ]:
# 1. Агрегація за датою та типом девайса
device_daily = df.groupby(['session_date', 'device_type'])['price'].sum().reset_index()

# 2. Побудова графіка
plt.figure(figsize=(14, 6))
palette_device = {'desktop': '#1f77b4', 'mobile': '#ff7f0e', 'tablet': '#2ca02c'}

ax = sns.lineplot(
    data=device_daily, 
    x='session_date', 
    y='price', 
    hue='device_type', 
    palette=palette_device,
    linewidth=2.2, 
    marker='o',
    markersize=4
)

plt.title('Динаміка продажів за типами девайсів (Desktop, Mobile, Tablet)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Загальні продажі ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Тип девайса', title_fontsize='11', fontsize='10')

# Форматування осі X: підписи кожні 7 днів
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%Y'))
plt.xticks(rotation=45)

# Форматування осі Y
ax.yaxis.set_major_formatter('{x:,.0f} $')

plt.tight_layout()
plt.show()

# 3. Зведена таблиця (перші 5 та останні 5 днів)
pivot_devices = device_daily.pivot(index='session_date', columns='device_type', values='price').fillna(0)
pivot_devices.index = pivot_devices.index.strftime('%Y-%m-%d')
pivot_devices_formatted = pivot_devices.apply(lambda col: col.map('{:,.2f} $'.format))

print("--- ПРОДАЖІ ЗА ТИПАМИ ДЕВАЙСІВ ПО ДНЯХ (Перші 5 та останні 5 днів) ---")
print(pd.concat([pivot_devices_formatted.head(5), pivot_devices_formatted.tail(5)]))

> **Інсайт по динаміці:** На тижневому графіку чітко простежуються два ключові піки — період Чорної П'ятниці (листопад) та максимальний передноворічний сплеск (грудень), після чого спостерігається стабілізація попиту в січні.

## Крок 5. Зведені таблиці (Pivot Tables)
Багатовимірний аналіз комбінацій каналів трафіку, пристроїв та категорій для оцінки перехресної конверсії та виручки.

In [ ]:
df_filtered_sessions = df[
    (df['traffic_channel'].notnull()) & 
    (df['traffic_channel'] != 'Undefined') & 
    (df['device_type'].notnull()) & 
    (df['device_type'] != 'Undefined')
]

pivot_sessions = pd.pivot_table(
    df_filtered_sessions,
    index='traffic_channel',
    columns='device_type',
    values='ga_session_id',
    aggfunc='nunique',
    margins=True,
    margins_name='Всього'
)

pivot_sessions_formatted = pivot_sessions.apply(lambda col: col.map('{:,.0f}'.format))

print("--- ЗВЕДЕНА ТАБЛИЦЯ: КІЛЬКІСТЬ СЕСІЙ ЗА КАНАЛАМИ ТРАФІКУ ТА ДЕВАЙСАМИ ---")
print(pivot_sessions_formatted)

In [ ]:
# ==========================================
# КРОК 5.2: Зведена таблиця: Продажі Топ-10 категорій у Топ-5 країнах
# ==========================================

In [ ]:
top5_countries = df.groupby('country')['price'].sum().nlargest(5).index.tolist()
top10_categories = df.groupby('product_category')['price'].sum().nlargest(10).index.tolist()

df_top_sales = df[df['country'].isin(top5_countries) & df['product_category'].isin(top10_categories)]

pivot_sales = pd.pivot_table(
    df_top_sales,
    index='product_category',
    columns='country',
    values='price',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Всього'
).sort_values(by='Всього', ascending=False)

country_cols = [c for c in top5_countries if c in pivot_sales.columns] + ['Всього']
pivot_sales = pivot_sales[country_cols]

pivot_sales_formatted = pivot_sales.apply(lambda col: col.map('{:,.2f} $'.format))

print("--- ЗВЕДЕНА ТАБЛИЦЯ: ПРОДАЖІ ТОП-10 КАТЕГОРІЙ У ТОП-5 КРАЇНАХ ---")
print(pivot_sales_formatted)

In [ ]:
# ------------------------------------------
# 1. Зведена таблиця: Середній чек за континентами та девайсами
# ------------------------------------------
pivot_avg_check = pd.pivot_table(
    df[df['price'].notnull()],
    index='continent',
    columns='device_type',
    values='price',
    aggfunc='mean'
).dropna(how='all')

pivot_avg_check_formatted = pivot_avg_check.apply(lambda col: col.map('{:,.2f} $'.format))

print("--- ВЛАСНА ЗВЕДЕНА ТАБЛИЦЯ 1: СЕРЕДНІЙ ЧЕК ЗА КОНТИНЕНТАМИ ТА ДЕВАЙСАМИ ---")
print(pivot_avg_check_formatted)


# ------------------------------------------
# 2. Зведена таблиця: Конверсія сесій у покупки (%) за каналами трафіку та континентами
# ------------------------------------------
pivot_conversion = pd.pivot_table(
    df,
    index='traffic_channel',
    columns='continent',
    values='price',
    aggfunc=lambda x: (x.notnull().sum() / len(x)) * 100
)

main_continents = ['Americas', 'Asia', 'Europe', 'Africa', 'Oceania']
pivot_conversion = pivot_conversion[[c for c in main_continents if c in pivot_conversion.columns]]

pivot_conversion_formatted = pivot_conversion.apply(lambda col: col.map('{:.2f} %'.format))

print("\n--- ВЛАСНА ЗВЕДЕНА ТАБЛИЦЯ 2: КОНВЕРСІЯ СЕСІЙ У ПОКУПКИ (%) ---")
print(pivot_conversion_formatted)

> **Підсумок зведених таблиць:** Найбільша концентрація чеків спостерігається у зв'язці Desktop + Organic/Paid Search при купівлі великих категорій меблів.

## Крок 6. Статистичний аналіз взаємозв'язків (Correlation & P-value)
Оцінка кореляційних залежностей між числовими змінними та перевірка їхньої статистичної значущості.

In [ ]:
daily_metrics = df.groupby('session_date').agg(
    total_sessions=('ga_session_id', 'nunique'),
    total_sales=('price', 'sum')
).reset_index()

r_pearson, p_pearson = stats.pearsonr(daily_metrics['total_sessions'], daily_metrics['total_sales'])
r_spearman, p_spearman = stats.spearmanr(daily_metrics['total_sessions'], daily_metrics['total_sales'])

plt.figure(figsize=(10, 6))
sns.regplot(
    data=daily_metrics, 
    x='total_sessions', 
    y='total_sales', 
    color='#1f77b4',
    scatter_kws={'s': 60, 'alpha': 0.8},
    line_kws={'color': '#e07a5f', 'linewidth': 2}
)

plt.title('Взаємозв\'язок між кількістю сесій та продажами за датами', fontsize=14, pad=15)
plt.xlabel('Кількість сесій за день', fontsize=12)
plt.ylabel('Загальні продажі ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()

print("--- СТАТИСТИЧНИЙ АНАЛІЗ ЗВ'ЯЗКУ ---")
print(f"Коефіцієнт кореляції Пірсона (r): {r_pearson:.4f}")
print(f"p-value (Пірсон): {p_pearson:.4e}")
print(f"Коефіцієнт кореляції Спірмена (rho): {r_spearman:.4f}")
print(f"p-value (Спірмен): {p_spearman:.4e}")

In [ ]:
top3_continents = ['Americas', 'Asia', 'Europe']
daily_continents_sales = df[df['continent'].isin(top3_continents)].pivot_table(
    index='session_date',
    columns='continent',
    values='price',
    aggfunc='sum'
).fillna(0)

corr_cont = daily_continents_sales.corr(method='pearson')

p_values_cont = pd.DataFrame(index=corr_cont.index, columns=corr_cont.columns)
for col1 in daily_continents_sales.columns:
    for col2 in daily_continents_sales.columns:
        _, p_val = stats.pearsonr(daily_continents_sales[col1], daily_continents_sales[col2])
        p_values_cont.loc[col1, col2] = p_val

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_cont, 
    annot=True, 
    fmt=".3f", 
    cmap='Blues', 
    vmin=0, 
    vmax=1, 
    linewidths=1,
    cbar_kws={'label': 'Коефіцієнт кореляції (r)'}
)

plt.title('Матриця кореляції продажів між Топ-3 континентами', fontsize=14, pad=15)
plt.tight_layout()

plt.show()

print("--- МАТРИЦЯ КОРЕЛЯЦІЇ (r) ---")
print(corr_cont.round(4))

print("\n--- МАТРИЦЯ P-VALUE ---")
print(p_values_cont.apply(lambda col: col.map('{:.4f}'.format)))

In [ ]:
main_channels = ['Organic Search', 'Paid Search', 'Direct', 'Social Search']
daily_channels_sales = df[df['traffic_channel'].isin(main_channels)].pivot_table(
    index='session_date',
    columns='traffic_channel',
    values='price',
    aggfunc='sum'
).fillna(0)

corr_chan = daily_channels_sales.corr(method='pearson')

p_values_chan = pd.DataFrame(index=corr_chan.index, columns=corr_chan.columns)
for col1 in daily_channels_sales.columns:
    for col2 in daily_channels_sales.columns:
        _, p_val = stats.pearsonr(daily_channels_sales[col1], daily_channels_sales[col2])
        p_values_chan.loc[col1, col2] = p_val

plt.figure(figsize=(9, 6.5))
sns.heatmap(
    corr_chan, 
    annot=True, 
    fmt=".3f", 
    cmap='Greens', 
    vmin=0, 
    vmax=1, 
    linewidths=1,
    cbar_kws={'label': 'Коефіцієнт кореляції (r)'}
)

plt.title('Матриця кореляції продажів між каналами трафіку', fontsize=14, pad=15)
plt.tight_layout()

plt.show()

print("--- МАТРИЦЯ КОРЕЛЯЦІЇ (r) ---")
print(corr_chan.round(4))

print("\n--- МАТРИЦЯ P-VALUE ---")
print(p_values_chan.apply(lambda col: col.map('{:.6f}'.format)))

In [ ]:
top5_cats = df.groupby('product_category')['price'].sum().nlargest(5).index.tolist()

daily_cats_sales = df[df['product_category'].isin(top5_cats)].pivot_table(
    index='session_date',
    columns='product_category',
    values='price',
    aggfunc='sum'
).fillna(0)

corr_cats = daily_cats_sales.corr(method='pearson')

p_values_cats = pd.DataFrame(index=corr_cats.index, columns=corr_cats.columns)
for col1 in daily_cats_sales.columns:
    for col2 in daily_cats_sales.columns:
        _, p_val = stats.pearsonr(daily_cats_sales[col1], daily_cats_sales[col2])
        p_values_cats.loc[col1, col2] = p_val

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_cats, 
    annot=True, 
    fmt=".3f", 
    cmap='Oranges', 
    vmin=0, 
    vmax=1, 
    linewidths=1,
    cbar_kws={'label': 'Коефіцієнт кореляції (r)'}
)

plt.title('Матриця кореляції продажів між Топ-5 категоріями товарів', fontsize=14, pad=15)
plt.tight_layout()

plt.show()

print("--- МАТРИЦЯ КОРЕЛЯЦІЇ (r) ---")
print(corr_cats.round(4))

print("\n--- МАТРИЦЯ P-VALUE ---")
print(p_values_cats.apply(lambda col: col.map('{:.6f}'.format)))

In [ ]:
daily_devices = df.pivot_table(
    index='session_date',
    columns='device_type',
    values='price',
    aggfunc='sum'
).fillna(0)

r_dev, p_dev = stats.pearsonr(daily_devices['desktop'], daily_devices['mobile'])

# 2. Формуємо дані для перевірки залежності між кількістю замовлень та середнім чеком (AOV)
daily_aov = df[df['price'].notnull()].groupby('session_date').agg(
    orders_count=('price', 'count'),
    aov=('price', 'mean')
).reset_index()

r_aov, p_aov = stats.pearsonr(daily_aov['orders_count'], daily_aov['aov'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.regplot(
    data=daily_devices, 
    x='desktop', 
    y='mobile', 
    ax=axes[0], 
    color='#2b5c8f',
    scatter_kws={'s': 50},
    line_kws={'color': '#e07a5f'}
)
axes[0].set_title(f'Desktop vs Mobile продажі\n(r = {r_dev:.3f}, p = {p_dev:.4f})', fontsize=12)
axes[0].set_xlabel('Продажі Desktop ($)')
axes[0].set_ylabel('Продажі Mobile ($)')
axes[0].grid(True, linestyle='--', alpha=0.5)

sns.regplot(
    data=daily_aov, 
    x='orders_count', 
    y='aov', 
    ax=axes[1], 
    color='#81b29a',
    scatter_kws={'s': 50},
    line_kws={'color': '#e07a5f'}
)
axes[1].set_title(f'Кількість замовлень vs Середній чек (AOV)\n(r = {r_aov:.3f}, p = {p_aov:.4f})', fontsize=12)
axes[1].set_xlabel('Кількість замовлень за день')
axes[1].set_ylabel('Середній чек AOV ($)')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("--- СТАТИСТИКА ДОДАТКОВИХ ВЗАЄМОЗВ'ЯЗКІВ ---")
print(f"1. Desktop vs Mobile: r = {r_dev:.4f}, p-value = {p_dev:.6f} (Статистично значуще)")
print(f"2. Замовлення vs AOV: r = {r_aov:.4f}, p-value = {p_aov:.4f} (НЕ значуще)")


## Крок 7. Статистичний аналіз відмінностей між групами (Hypothesis Testing)
Порівняння середнього чека покупки (`price`) між зареєстрованими користувачами (`user_id.notna()`) та гостями сайту (`user_id.isna()`).

In [ ]:
df_orders_user = df[df['price'].notnull()].copy()

reg_orders = df_orders_user[df_orders_user['user_id'].notnull()]['price']
non_reg_orders = df_orders_user[df_orders_user['user_id'].isnull()]['price']

mwu_stat, p_value_mwu = stats.mannwhitneyu(
    reg_orders, 
    non_reg_orders, 
    alternative='two-sided'
)

plt.figure(figsize=(8, 5.5))
df_orders_user['user_status'] = df_orders_user['user_id'].apply(lambda x: 'Зареєстровані' if pd.notnull(x) else 'Незареєстровані')

sns.boxplot(
    data=df_orders_user, 
    x='user_status', 
    y='price', 
    hue='user_status',
    palette=['#81b29a', '#e07a5f'], 
    legend=False,
    width=0.4
)

plt.title('Розподіл вартості замовлення: Зареєстровані vs Незареєстровані', fontsize=13, pad=15, fontweight='bold')
plt.xlabel('Статус користувача', fontsize=11)
plt.ylabel('Вартість замовлення ($)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 4. Вивід статистик
print("--- СЕРЕДНІ ЧЕКИ ТА МЕДІАНИ (AOV) ---")
print(f"Зареєстровані:   N = {len(reg_orders):,}, Середнє = {reg_orders.mean():,.2f} $, Медіана = {reg_orders.median():,.2f} $")
print(f"Незареєстровані: N = {len(non_reg_orders):,}, Середнє = {non_reg_orders.mean():,.2f} $, Медіана = {non_reg_orders.median():,.2f} $\n")

print("--- КРИТЕРІЙ МАННА-УЇТНІ (Рівень замовлень) ---")
print(f"U-статистика: {mwu_stat}")
print(f"p-value: {p_value_mwu:.4e}")

alpha = 0.05
if p_value_mwu < alpha:
    print(f"\nВисновок: p-value < {alpha}. Різниця у середньому чеку між групами є статистично значущою.")
else:
    print(f"\nВисновок: p-value >= {alpha}. Статистично значущої різниці у середньому чеку не виявлено.")

In [ ]:
channels_list = ['Organic Search', 'Paid Search', 'Direct', 'Social Search']
daily_sessions_channel = df[df['traffic_channel'].isin(channels_list)].pivot_table(
    index='session_date',
    columns='traffic_channel',
    values='ga_session_id',
    aggfunc='nunique'
).fillna(0)

stat_kw, p_val_kw = stats.kruskal(
    daily_sessions_channel['Organic Search'],
    daily_sessions_channel['Paid Search'],
    daily_sessions_channel['Direct'],
    daily_sessions_channel['Social Search']
)

plt.figure(figsize=(9, 5.5))
sns.boxplot(data=daily_sessions_channel, palette='Set2', width=0.5)

plt.title('Розподіл щоденної кількості сесій за каналами трафіку', fontsize=13, pad=15)
plt.ylabel('Кількість сесій на день', fontsize=11)
plt.xlabel('Канал трафіку', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("--- СЕРЕДНЯ ЩОДЕННА КІЛЬКІСТЬ СЕСІЙ ---")
print(daily_sessions_channel.mean().round(2).to_string())

print("\n--- ТЕСТ КРАСКЕЛА-УОЛЛІСА (Kruskal-Wallis H-test) ---")
print(f"H-статистика: {stat_kw:.4f}")
print(f"p-value: {p_val_kw:.4e}")

In [ ]:
df_eu_am = df[df['continent'].isin(['Europe', 'Americas'])].copy()

df_eu_am['is_organic'] = df_eu_am['traffic_channel'] == 'Organic Search'

contingency_table = pd.crosstab(df_eu_am['continent'], df_eu_am['is_organic'])
contingency_table.columns = ['Інший трафік', 'Organic Search']

chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)

organic_pct = (contingency_table['Organic Search'] / contingency_table.sum(axis=1) * 100).round(2)

print("--- ТАБЛИЦЯ СПРЯЖЕНОСТІ СЕСІЙ ---")
print(contingency_table)

print("\n--- ЧАСТКА ОРГАНІЧНОГО ТРАФІКУ (%) ---")
print(organic_pct.to_string())

print("\n--- КРИТЕРІЙ ХІ-КВАДРАТ (Chi-Square Test) ---")
print(f"Chi2-статистика: {chi2_stat:.4f}")
print(f"p-value: {p_val_chi2:.4f}")

In [ ]:
df_orders_dev = df[df['device_type'].isin(['desktop', 'mobile']) & df['price'].notnull()].copy()

desktop_orders = df_orders_dev[df_orders_dev['device_type'] == 'desktop']['price']
mobile_orders = df_orders_dev[df_orders_dev['device_type'] == 'mobile']['price']

mwu_dev_stat, p_val_dev_mwu = stats.mannwhitneyu(desktop_orders, mobile_orders, alternative='two-sided')

plt.figure(figsize=(8, 5.5))
sns.boxplot(
    data=df_orders_dev, 
    x='device_type', 
    y='price', 
    hue='device_type',
    palette=['#1f77b4', '#ff7f0e'], 
    legend=False,
    width=0.4
)

plt.xticks([0, 1], ['Desktop', 'Mobile'])
plt.title('Розподіл вартості замовлення: Desktop vs Mobile', fontsize=13, pad=15)
plt.xlabel('Тип девайса', fontsize=11)
plt.ylabel('Вартість замовлення ($)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("--- СЕРЕДНІ ЧЕКИ ТА МЕДІАНИ ---")
print(f"Desktop: Середнє = {desktop_orders.mean():,.2f} $, Медіана = {desktop_orders.median():,.2f} $")
print(f"Mobile:  Середнє = {mobile_orders.mean():,.2f} $, Медіана = {mobile_orders.median():,.2f} $\n")

print("--- КРИТЕРІЙ МАННА-УЇТНІ ---")
print(f"U-статистика: {mwu_dev_stat}")
print(f"p-value: {p_val_dev_mwu:.4f}")

> **Результат тестування:** Перевірка вибірок цін на рівні окремих замовлень дозволила оцінити різницю поведінки двох сегментів користувачів за середнім чеком (AOV).

## Крок 8. Інтерактивний дашборд у Tableau Public та загальні рекомендації

📊 **[Посилання на інтерактивний Tableau Public Dashboard](https://public.tableau.com/app/profile/mykyta.yemelianov3525/viz/Book1_17583094938840/UserTrafficDashboard?publish=yes)**

---

### 📌 Загальний бізнес-висновок (Executive Summary):
1. **Сезонність:** Пікові продажі формуються в кінці листопада та в середині грудня.
2. **Товарний фокус:** Категорії м'яких меблів (*Sofas & Armchairs*) та ринок США забезпечують основу фінансового результату.
3. **Рекомендації:**
   * Посилити роботу з базою зареєстрованих користувачів через персональні пропозиції перед піковими періодами.
   * Провести аудит мобільного оформлення замовлення, щоб скоротити відставання Mobile від Desktop за сумою чека.

In [ ]:
export_filename = 'portfolio_dataset_tableau.csv'

df.to_csv(export_filename, index=False)

print(f"--- ФАЙЛ УСПІШНО ЗБЕРЕЖЕНО ---")
print(f"Назва файлу: {export_filename}")
print(f"Кількість рядків: {len(df):,}")
print(f"Кількість колонок: {len(df.columns)}")
print(f"\nПерелік колонок для Tableau:\n{list(df.columns)}")